In [22]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict

In [23]:
load_dotenv()

True

In [24]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

In [25]:
# Create a state
class LLMState(TypedDict):

    question: str
    answer: str

In [29]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from start
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to LLM
    chain = llm | StrOutputParser()

    # update the answer in the state
    state['answer'] = chain.invoke(prompt)

    return state

In [30]:
# Create our graph
graph = StateGraph(LLMState)

# Add nodes
graph.add_node('llm_qa', llm_qa)

# Add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# Compile
workflow = graph.compile()

In [32]:
# execute
initial_state = {'question': 'how far is moon from earth?'}
final_state = workflow.invoke(initial_state)
print(final_state)
print(final_state['answer'])

{'question': 'how far is moon from earth?', 'answer': 'The average distance from the Earth to the Moon is about **238,855 miles** (384,400 kilometers). \n\nBecause the Moon travels in an elliptical (oval-shaped) orbit rather than a perfect circle, that distance changes constantly:\n* **Closest point (Perigee):** About 226,000 miles (363,300 km)\n* **Farthest point (Apogee):** About 251,000 miles (405,500 km)\n\nTo put that distance into perspective, you could fit all the planets in our solar system side-by-side in the space between the Earth and the Moon!'}
The average distance from the Earth to the Moon is about **238,855 miles** (384,400 kilometers). 

Because the Moon travels in an elliptical (oval-shaped) orbit rather than a perfect circle, that distance changes constantly:
* **Closest point (Perigee):** About 226,000 miles (363,300 km)
* **Farthest point (Apogee):** About 251,000 miles (405,500 km)

To put that distance into perspective, you could fit all the planets in our solar 